In [8]:
!pip install transformers[sentencepiece] datasets scikit-learn tqdm matplotlib seaborn

import os
import re
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel 
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "xlm-roberta-base"

if not os.path.exists('xlm_roberta_checkpoints/plots'):
    os.makedirs('xlm_roberta_checkpoints/plots')

print(f"Zariadenie: {device}")

Zariadenie: cuda


In [9]:
df = pd.read_csv('labeled_data.csv')

def clean_text(text):
    text = text.lower()
    text = re.sub(r'@[\w\-]+', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'\brt\b', '', text)
    text = re.sub(r'&amp;|&#\d+;', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_tweet'] = df['tweet'].apply(clean_text)
df = df.drop_duplicates(subset=['clean_tweet'])
df = df[df['clean_tweet'] != ''].dropna()

X = df['clean_tweet']
y = df['class']

print(f"Záznamov pre XLM-RoBERTa: {len(df)}")

Záznamov pre XLM-RoBERTa: 24443


In [10]:
class SimpleDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts = texts.values
        self.labels = labels.values
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text, 
            padding='max_length', 
            truncation=True, 
            max_length=64, 
            return_tensors='pt'
        )
        return {
            'ids': encoding['input_ids'].flatten(),
            'mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [11]:
class XLMRobertaClassifier(nn.Module):
    
    def __init__(self, num_classes=3):
        super(XLMRobertaClassifier, self).__init__()
        self.roberta = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.roberta.config.hidden_size, num_classes)

    def forward(self, ids, mask):
        outputs = self.roberta(input_ids=ids, attention_mask=mask)
        last_hidden_state = outputs.last_hidden_state
        cls_output = last_hidden_state[:, 0, :] 
        out = self.dropout(cls_output)
        return self.classifier(out)

In [12]:
def save_fold_loss_curve(history, fold_num):
    
    plt.figure(figsize=(4, 2.5))
    plt.plot(history['train'], label='Train Loss', linewidth=1.5, marker='o', markersize=4)
    plt.plot(history['val'], label='Val Loss', linewidth=1.5, marker='s', markersize=4)
    plt.title(f'XLM-R Loss: Fold {fold_num}', fontsize=9)
    plt.xlabel('Epocha', fontsize=8)
    plt.ylabel('Loss', fontsize=8)
    plt.xticks(range(len(history['train'])), range(1, len(history['train']) + 1), fontsize=7)
    plt.legend(fontsize=7)
    plt.grid(True, linestyle='--', alpha=0.5)
    
    plot_path = f'xlm_roberta_checkpoints/plots/fold_{fold_num}_loss_curves.png'
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.close()

In [13]:
def train_model_advanced(model, train_loader, val_loader, fold, epochs=5, patience=2):
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
    criterion = torch.nn.CrossEntropyLoss()
    
    best_val_loss = float('inf')
    epochs_no_improve = 0  
    history = {'train': [], 'val': []}

    print(f"Trénovanie XLM-RoBERTa Fold {fold}")
    
    for epoch in range(epochs):
        model.train()
        total_train_loss = 0
        loop = tqdm(train_loader, desc=f"Fold {fold} | Ep {epoch+1}")
        
        for batch in loop:
            optimizer.zero_grad()
            ids, mask = batch['ids'].to(device), batch['mask'].to(device)
            labels = batch['label'].to(device)
            outputs = model(ids, mask)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()

        model.eval()
        total_val_loss, correct = 0, 0
        with torch.no_grad():
            for batch in val_loader:
                ids, mask = batch['ids'].to(device), batch['mask'].to(device)
                labels = batch['label'].to(device)
                outputs = model(ids, mask)
                total_val_loss += criterion(outputs, labels).item()
                correct += (torch.argmax(outputs, dim=1) == labels).sum().item()

        avg_train = total_train_loss / len(train_loader)
        avg_val = total_val_loss / len(val_loader)
        history['train'].append(avg_train)
        history['val'].append(avg_val)

        print(f"[EPOCHA {epoch+1}] Train: {avg_train:.4f} | Val: {avg_val:.4f} | Acc: {correct/len(val_loader.dataset):.4f}")

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            torch.save(model.state_dict(), f"xlm_roberta_checkpoints/best_model_fold_{fold}.pt")
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
        
        if epochs_no_improve >= patience:
            print("Early Stopping!")
            break
    return history

In [14]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
tokenizer = AutoTokenizer.from_pretrained(model_name)

global_best_loss = float('inf')
best_val_indices = None

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n==================== XLM-R FOLD {fold+1} ====================")
    
    train_loader = DataLoader(SimpleDataset(X.iloc[train_idx], y.iloc[train_idx], tokenizer), batch_size=16, shuffle=True)
    val_loader = DataLoader(SimpleDataset(X.iloc[val_idx], y.iloc[val_idx], tokenizer), batch_size=16)
    
    model = XLMRobertaClassifier().to(device).float()
    history = train_model_advanced(model, train_loader, val_loader, fold=fold+1)
    save_fold_loss_curve(history, fold_num=fold+1)
    
    path_to_current_best = f"xlm_roberta_checkpoints/best_model_fold_{fold+1}.pt"
    model.load_state_dict(torch.load(path_to_current_best))
    
    model.eval()
    current_fold_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            logits = model(batch['ids'].to(device), batch['mask'].to(device))
            loss = torch.nn.CrossEntropyLoss()(logits, batch['label'].to(device))
            current_fold_loss += loss.item()
    
    avg_current_fold_loss = current_fold_loss / len(val_loader)
    print(f"Finálna strata pre Fold {fold+1}: {avg_current_fold_loss:.4f}")
    
    if avg_current_fold_loss < global_best_loss:
        global_best_loss = avg_current_fold_loss
        najlepsi_fold_cislo = fold + 1
        best_val_indices = val_idx
        torch.save(model.state_dict(), "xlm_roberta_checkpoints/absolute_best_xlmr_model.pt")
        print(f"--> Nový absolútne najlepší model nájdený vo Folde {fold+1}!")


==================== XLM-R FOLD 1 ====================


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Trénovanie XLM-RoBERTa Fold 1


Fold 1 | Ep 1:   0%|          | 0/1223 [00:00<?, ?it/s]

...

[EPOCHA 3] Train: 0.2350 | Val: 0.2551 | Acc: 0.9094


Fold 5 | Ep 4:   0%|          | 0/1223 [00:00<?, ?it/s]

[EPOCHA 4] Train: 0.2154 | Val: 0.2705 | Acc: 0.9086


Fold 5 | Ep 5:   0%|          | 0/1223 [00:00<?, ?it/s]

[EPOCHA 5] Train: 0.1892 | Val: 0.2737 | Acc: 0.9073
Early Stopping!
Finálna strata pre Fold 5: 0.2551


In [15]:
def get_detailed_report(model, loader, title="VÝSLEDOK"):
    
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            logits = model(batch['ids'].to(device), batch['mask'].to(device))
            all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            all_labels.extend(batch['label'].numpy())
    
    print(f"\n=== {title} ===")
    print(classification_report(all_labels, all_preds, target_names=['Hate', 'Offensive', 'Neither'], digits=4))
    return all_labels, all_preds

final_model = XLMRobertaClassifier().to(device)
final_model.load_state_dict(torch.load("xlm_roberta_checkpoints/absolute_best_xlmr_model.pt"))
test_loader = DataLoader(SimpleDataset(X.iloc[best_val_indices], y.iloc[best_val_indices], tokenizer), batch_size=16)

y_true, y_pred = get_detailed_report(final_model, test_loader, "FINÁLNY REPORT: XLM-RoBERTa")



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



=== FINÁLNY REPORT: XLM-RoBERTa ===
              precision    recall  f1-score   support

        Hate     0.5876    0.2021    0.3008       282
   Offensive     0.9408    0.9699    0.9551      3786
     Neither     0.8695    0.9415    0.9041       821

    accuracy                         0.9208      4889
   macro avg     0.7993    0.7045    0.7200      4889
weighted avg     0.9085    0.9208    0.9088      4889

